# Test BayesCard

本 notebook 用于将 BayesCard（纯 BN 路径）接入现有实验体系。

- 复用 `experiment/BayesCardRunner.py` 进行训练与推理编排
- 输出到 `experiment/checkpoint/BayesCard/`
- 对四个 benchmark 先做可行性评估，再仅执行可支持项

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = (Path.cwd().parent if (Path.cwd().parent / 'Benchmark').exists() else Path.cwd()).resolve()
EXPERIMENT_DIR = PROJECT_ROOT / 'experiment'
CHECKPOINT_DIR = EXPERIMENT_DIR / 'checkpoint' / 'BayesCard'

sys.path.insert(0, str(EXPERIMENT_DIR))
from BayesCardRunner import BayesCardRunner

runner = BayesCardRunner(PROJECT_ROOT)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'CHECKPOINT_DIR={CHECKPOINT_DIR}')

/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.4) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


PROJECT_ROOT=/home/liwei/starCE
CHECKPOINT_DIR=/home/liwei/starCE/experiment/checkpoint/BayesCard


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/site-packages/pgmpy/utils/utils.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
ALL_BENCHMARKS = ['STATS', 'JOBLight', 'JOBLightRanges', 'JOBM', 'StatsJoin']
SEED = 0

capability_rows = []
for benchmark in ALL_BENCHMARKS:
    supported, reason = runner.evaluate_benchmark_capability(benchmark)
    capability_rows.append(
        {
            'benchmark': benchmark,
            'supported_by_pure_bayescard_bn': supported,
            'reason': reason,
        }
    )

capability_df = pd.DataFrame(capability_rows)
capability_df

,benchmark,supported_by_pure_bayescard_bn,reason
0,STATS-CEB,True,支持：可走 methods/SafeBound/bayescard 的纯 BayesCard...
1,JOBLight,True,支持：可走 methods/SafeBound/bayescard 的纯 BayesCard...
2,JOBLightRanges,True,支持：可走 methods/SafeBound/bayescard 的纯 BayesCard...
3,JOBM,False,不支持纯 BayesCard：当前仓库中 JOBM 的可靠路径依赖采样/额外处理，并非纯 B...
4,StatsJoin,True,支持：复用 STATS BN 模型，仅推理无需训练。


## Execute

下面仅执行可支持的 benchmark（`STATS`、`JOBLight`、`JOBLightRanges`、`StatsJoin`）。

`JOBM` 在当前仓库中可靠路径依赖采样，不属于纯 BayesCard/BN，本 notebook 不执行。
`StatsJoin` 复用 STATS BN 模型，仅推理不训练。

In [3]:
BENCHMARKS_TO_RUN = [
    row['benchmark']
    for row in capability_rows
    if row['supported_by_pure_bayescard_bn']
]

summary_df = runner.run_all(BENCHMARKS_TO_RUN, seed=SEED, force_retrain=True)
summary_df

INFO:test_benchmark:Loaded BN model: 0_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 10_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 1_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 2_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 3_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 4_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 5_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 6_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 7_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 8_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 9_chow-liu_1.pkl
INFO:test_benchmark:Loaded 11 BN models from /home/liwei/starCE/experiment/checkpoint/BayesCard/models/stats
INFO:test_benchmark:Loaded BN model: 0_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 1_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 2_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 3_chow-liu_1.pkl
INFO:test_benchmark:Loaded BN model: 4_chow-liu_1.pkl
INFO:test_benchmark:Loaded

,benchmark,seed,preprocess_time_sec,train_time_sec,total_build_like_time_sec,total_eval_like_time_sec,query_count,avg_latency_sec,checkpoint_output,model_dir,statistics_size_bytes
0,STATS,0,27.228475,0.0,27.228475,39.034821,2471,0.012796,/home/liwei/starCE/experiment/checkpoint/Bayes...,/home/liwei/starCE/experiment/checkpoint/Bayes...,6322830
1,JOBLight,0,0.000000,0.0,0.000000,2.886187,451,0.004063,/home/liwei/starCE/experiment/checkpoint/Bayes...,/home/liwei/starCE/experiment/checkpoint/Bayes...,1628974
2,JOBLightRanges,0,32.657151,0.0,32.657151,66.357374,8292,0.005744,/home/liwei/starCE/experiment/checkpoint/Bayes...,/home/liwei/starCE/experiment/checkpoint/Bayes...,2308592
3,STATSJOIN,0,0.000000,0.0,0.000000,1.949156,226,0.006311,/home/liwei/starCE/experiment/checkpoint/Bayes...,/home/liwei/starCE/experiment/checkpoint/Bayes...,6322830


In [4]:
summary_csv = CHECKPOINT_DIR / 'benchmark_times.csv'
summary_df = pd.read_csv(summary_csv)

print(f'Summary CSV: {summary_csv}')
for _, row in summary_df.iterrows():
    print(f"{row['benchmark']}: card={row['checkpoint_output']}, eval_time={row['total_eval_like_time_sec']}")

summary_df

Summary CSV: /home/liwei/starCE/experiment/checkpoint/BayesCard/benchmark_times.csv
STATS-CEB: card=/home/liwei/starCE/experiment/checkpoint/BayesCard/card_stats.txt, eval_time=19.241838455200195
STATS: card=/home/liwei/starCE/experiment/checkpoint/BayesCard/card_stats.txt, eval_time=39.03482127189636
JOBLight: card=/home/liwei/starCE/experiment/checkpoint/BayesCard/card_joblight.txt, eval_time=2.8861870765686035
JOBLightRanges: card=/home/liwei/starCE/experiment/checkpoint/BayesCard/card_joblr.txt, eval_time=66.35737371444702
STATSJOIN: card=/home/liwei/starCE/experiment/checkpoint/BayesCard/card_statsjoin.txt, eval_time=1.9491560459136963


,benchmark,total_eval_like_time_sec,checkpoint_output,train_time_sec,total_build_like_time_sec,preprocess_time_sec,query_count,avg_latency_sec,model_dir,statistics_size_bytes,seed,model_path
0,STATS-CEB,19.241838,/home/liwei/starCE/experiment/checkpoint/Bayes...,36.990183,36.990183,0.000000,2471,0.007779,NaN,6300088,0,/home/liwei/starCE/methods/FactorJoin/checkpoi...
1,STATS,39.034821,/home/liwei/starCE/experiment/checkpoint/Bayes...,0.000000,27.228475,27.228475,2471,0.012796,/home/liwei/starCE/experiment/checkpoint/Bayes...,6322830,0,NaN
2,JOBLight,2.886187,/home/liwei/starCE/experiment/checkpoint/Bayes...,0.000000,0.000000,0.000000,451,0.004063,/home/liwei/starCE/experiment/checkpoint/Bayes...,1628974,0,NaN
3,JOBLightRanges,66.357374,/home/liwei/starCE/experiment/checkpoint/Bayes...,0.000000,32.657151,32.657151,8292,0.005744,/home/liwei/starCE/experiment/checkpoint/Bayes...,2308592,0,NaN
4,STATSJOIN,1.949156,/home/liwei/starCE/experiment/checkpoint/Bayes...,0.000000,0.000000,0.000000,226,0.006311,/home/liwei/starCE/experiment/checkpoint/Bayes...,6322830,0,NaN
